# Milestone 5: Proactive Credit Limit Increases

## 1. Load the Existing Borrowers and Early Warning Results

This milestone identifies existing borrowers who may qualify for an
illustrative credit limit increase.

We will:
- Check repayment behaviour and credit utilization.
- Use the early warning results to screen for deterioration risk.
- Calculate a proposed increase within explicit limits.
- Assess the proposal under stated profitability assumptions.

There is no observed limit-increase outcome in our synthetic data.
Therefore, this is a policy simulation rather than a trained
limit-increase prediction model.

We begin with the held-out accounts from Milestone 4. Their early warning
predictions were generated without using these accounts to fit that model.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

onbook_path = root / "data" / "raw" / "sme_onbook_portfolio.csv"
ews_path = root / "reports" / "sme_ews_v1" / "test_scored.csv"

for path in [onbook_path, ews_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing input file: {path}")

onbook = pd.read_csv(
    onbook_path,
    dtype={"applicant_id": "string"}
)

ews_scores = pd.read_csv(
    ews_path,
    dtype={"applicant_id": "string"}
)

for name, frame in [
    ("Booked portfolio", onbook),
    ("Held-out early warning results", ews_scores),
]:
    assert frame["applicant_id"].notna().all(), f"{name}: missing IDs."
    assert frame["applicant_id"].is_unique, f"{name}: duplicate IDs."

assert set(ews_scores["applicant_id"]).issubset(
    set(onbook["applicant_id"])
), "Some early warning accounts are missing from the booked portfolio."

assert ews_scores["predicted_deterioration"].between(0, 1).all()

print(f"Booked accounts available: {len(onbook):,}")
print(f"Held-out accounts for this simulation: {len(ews_scores):,}")

print("\nBooked-portfolio columns:")
print(onbook.columns.tolist())

print("\nEarly warning result columns:")
print(ews_scores.columns.tolist())

print("\nInput checks passed.")

Booked accounts available: 9,887
Held-out accounts for this simulation: 2,967

Booked-portfolio columns:
['applicant_id', 'industry', 'region', 'entity_type', 'years_in_business', 'employees', 'annual_revenue', 'revenue_growth_12m', 'profit_margin', 'net_income', 'total_debt', 'current_ratio', 'dscr', 'leverage', 'bureau_score', 'credit_history_months', 'prior_delinquencies_12m', 'utilization', 'public_records', 'requested_amount', 'collateral_flag', 'credit_limit', 'current_balance', 'utilization_onbook', 'utilization_trend_24m', 'tenure_months', 'max_days_past_due', 'months_delinquent_24m', 'overdrafts_24m', 'average_deposit_inflow', 'default_12m', 'deterioration_next_6_12mo']

Early warning result columns:
['review_rank', 'applicant_id', 'utilization_recent_3m', 'utilization_change', 'utilization_std_24m', 'deposit_recent_3m', 'deposit_decline_pct', 'max_dpd_24m', 'max_dpd_recent_3m', 'delinquent_months_24m', 'overdrafts_24m', 'overdrafts_recent_6m', 'application_leverage', 'applica

## 2. Assemble the Credit Limit Review Dataset

For each held-out borrower, combine:
- Early warning predictions and behavioural features.
- Credit limit and outstanding balance from month 23.
- Annual revenue recorded at application.

Application revenue is historical, not a refreshed financial statement.
Any offers calculated from it will therefore be illustrative and subject
to updated financial information.

Default and deterioration outcomes are excluded from the decision dataset.

In [2]:
panel_path = root / "data" / "raw" / "sme_behavioural_panel.csv"

panel = pd.read_csv(
    panel_path,
    dtype={"applicant_id": "string"}
)

# Latest observed month: use balances available at the review date.
review_month = 23

latest_snapshot = (
    panel.loc[
        panel["month_index"].eq(review_month),
        ["applicant_id", "credit_limit", "outstanding_balance"]
    ]
    .rename(columns={
        "credit_limit": "current_credit_limit",
        "outstanding_balance": "latest_outstanding_balance"
    })
    .copy()
)

assert latest_snapshot["applicant_id"].is_unique

application_context = (
    onbook[["applicant_id", "annual_revenue"]]
    .rename(columns={
        "annual_revenue": "application_annual_revenue"
    })
)

limit_review = (
    ews_scores
    .merge(
        latest_snapshot,
        on="applicant_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        application_context,
        on="applicant_id",
        how="left",
        validate="one_to_one"
    )
)

required_columns = [
    "current_credit_limit",
    "latest_outstanding_balance",
    "application_annual_revenue"
]

assert len(limit_review) == len(ews_scores)
assert limit_review[required_columns].notna().all().all()
assert limit_review["current_credit_limit"].gt(0).all()
assert limit_review["latest_outstanding_balance"].ge(0).all()
assert limit_review["application_annual_revenue"].gt(0).all()

# Outcomes must not enter offer decisions.
assert {
    "default_12m",
    "deterioration_next_6_12mo"
}.isdisjoint(limit_review.columns)

limit_review["latest_utilization"] = (
    limit_review["latest_outstanding_balance"]
    / limit_review["current_credit_limit"]
)

print(f"Accounts ready for review: {len(limit_review):,}")
print(f"Balance snapshot: month {review_month}")
print("Join and decision-input checks passed.")

display(
    limit_review[[
        "applicant_id",
        "current_credit_limit",
        "latest_outstanding_balance",
        "latest_utilization",
        "predicted_deterioration"
    ]].head().style.format({
        "current_credit_limit": "{:,.0f}",
        "latest_outstanding_balance": "{:,.0f}",
        "latest_utilization": "{:.2%}",
        "predicted_deterioration": "{:.2%}"
    })
)

Accounts ready for review: 2,967
Balance snapshot: month 23
Join and decision-input checks passed.


,applicant_id,current_credit_limit,latest_outstanding_balance,latest_utilization,predicted_deterioration
0,SME003868,"6,488,000","8,110,000",125.00%,78.36%
1,SME010564,"1,485,000","1,307,900",88.07%,76.91%
2,SME005489,"9,942,000","12,427,500",125.00%,75.85%
3,SME003202,"4,678,000","5,828,300",124.59%,72.70%
4,SME002367,"614,000","636,300",103.63%,71.75%


## 3. Screen Borrowers for a Possible Limit Increase

A borrower passes the initial screen when all conditions below are met:

| Check | Illustrative rule | Purpose |
|---|---|---|
| Early warning risk | Predicted deterioration ≤ 10% | Avoid borrowers showing elevated risk |
| Credit usage | Recent three-month average utilization between 50% and 85% | Identify borrowers using their existing facility |
| Latest credit usage | Latest utilization ≤ 90% | Exclude accounts currently close to or above their limit |
| Recent repayments | No days past due in the last three months | Require recent payment discipline |
| Deposit inflows | Decline no greater than 10% | Avoid materially weakening cash inflows |
| Overdrafts | At most two in the last six months | Limit signs of cash-flow stress |
| Utilization trend | Increase no greater than 15 percentage points | Avoid rapidly rising credit dependence |

These are initial screening assumptions, not validated lending rules.
The deterioration estimate is not a default PD and will not be substituted
for PD in profitability calculations.

Passing this screen does not yet produce an offer. We must still calculate
available headroom and assess the economics.

In [3]:
eligibility_rules = {
    "Low early warning risk":
        limit_review["predicted_deterioration"].le(0.10),

    "Meaningful, moderate credit usage":
        limit_review["utilization_recent_3m"].between(0.50, 0.85),

    "Latest utilization within limit":
        limit_review["latest_utilization"].le(0.90),

    "No recent missed payments":
        limit_review["max_dpd_recent_3m"].eq(0),

    "Stable deposit inflows":
        limit_review["deposit_decline_pct"].le(0.10),

    "Limited recent overdrafts":
        limit_review["overdrafts_recent_6m"].le(2),

    "No sharp utilization increase":
        limit_review["utilization_change"].le(0.15)
}

eligibility_checks = pd.DataFrame(
    eligibility_rules,
    index=limit_review.index
)

assert eligibility_checks.notna().all().all()

limit_review["passes_initial_screen"] = eligibility_checks.all(axis=1)

limit_review["screen_failure_reasons"] = eligibility_checks.apply(
    lambda row: "; ".join(row.index[~row])
    if not row.all() else "Passed initial screen",
    axis=1
)

# Show how many remain after applying each rule in sequence.
remaining = pd.Series(True, index=limit_review.index)
screening_steps = []

for rule_name in eligibility_checks.columns:
    remaining = remaining & eligibility_checks[rule_name]

    screening_steps.append({
        "rule_added": rule_name,
        "accounts_remaining": int(remaining.sum())
    })

screening_summary = pd.DataFrame(screening_steps)

display(screening_summary)

print(
    f"\nPassed initial screen: "
    f"{limit_review['passes_initial_screen'].sum():,} "
    f"of {len(limit_review):,} accounts"
)
print("These are candidates for further assessment, not final offers.")

,rule_added,accounts_remaining
0,Low early warning risk,601
1,"Meaningful, moderate credit usage",105
2,Latest utilization within limit,69
3,No recent missed payments,67
4,Stable deposit inflows,42
5,Limited recent overdrafts,35
6,No sharp utilization increase,22



Passed initial screen: 22 of 2,967 accounts
These are candidates for further assessment, not final offers.


## 4. Calculate Provisional Credit Limit Increases

For borrowers passing the initial screen, the provisional increase is
the smallest of:

1. The increase needed to bring recent average usage to 65% of the new limit.
2. 25% of the current credit limit.
3. Available headroom before the total limit reaches 30% of application revenue.

Recent average balance is calculated from the monthly behavioural panel.

Amounts are rounded down to the nearest ₹10,000. If there is no positive
headroom, no increase is proposed.

These are illustrative sizing assumptions. The revenue cap is a screening
proxy, not a complete affordability assessment. Profitability and refreshed
financial information still need consideration.

In [4]:
import numpy as np

# Average actual balance over the latest three observed months.
recent_balances = (
    panel.loc[panel["month_index"].between(21, 23)]
    .groupby("applicant_id")["outstanding_balance"]
    .mean()
)

limit_review["average_balance_recent_3m"] = (
    limit_review["applicant_id"].map(recent_balances)
)

assert limit_review["average_balance_recent_3m"].notna().all()

# Explicit assumptions for this simulation.
TARGET_UTILIZATION = 0.65
MAX_INCREASE_SHARE = 0.25
REVENUE_LIMIT_SHARE = 0.30
ROUNDING_UNIT = 10_000

limit_review["usage_based_headroom"] = (
    limit_review["average_balance_recent_3m"] / TARGET_UTILIZATION
    - limit_review["current_credit_limit"]
).clip(lower=0)

limit_review["maximum_increase_amount"] = (
    limit_review["current_credit_limit"] * MAX_INCREASE_SHARE
)

limit_review["revenue_based_headroom"] = (
    limit_review["application_annual_revenue"] * REVENUE_LIMIT_SHARE
    - limit_review["current_credit_limit"]
).clip(lower=0)

headroom = limit_review[[
    "usage_based_headroom",
    "maximum_increase_amount",
    "revenue_based_headroom"
]].min(axis=1)

rounded_increase = (
    np.floor(headroom / ROUNDING_UNIT) * ROUNDING_UNIT
)

limit_review["provisional_increase"] = rounded_increase.where(
    limit_review["passes_initial_screen"],
    0.0
)

limit_review["provisional_new_limit"] = (
    limit_review["current_credit_limit"]
    + limit_review["provisional_increase"]
)

increase_candidates = limit_review.loc[
    limit_review["provisional_increase"].gt(0)
].copy()

assert (
    limit_review["provisional_increase"]
    <= limit_review["maximum_increase_amount"] + 1e-6
).all()

assert increase_candidates["passes_initial_screen"].all()

assert (
    increase_candidates["provisional_new_limit"]
    <= (
        increase_candidates["application_annual_revenue"]
        * REVENUE_LIMIT_SHARE
    ) + 1e-6
).all()

print(
    f"Passed initial screen: "
    f"{limit_review['passes_initial_screen'].sum():,}"
)
print(f"Candidates with a positive increase: {len(increase_candidates):,}")
print(
    f"Total provisional additional limit: "
    f"₹{increase_candidates['provisional_increase'].sum():,.0f}"
)

display(
    increase_candidates[[
        "applicant_id",
        "predicted_deterioration",
        "current_credit_limit",
        "average_balance_recent_3m",
        "provisional_increase",
        "provisional_new_limit"
    ]].head(10).style.format({
        "predicted_deterioration": "{:.2%}",
        "current_credit_limit": "{:,.0f}",
        "average_balance_recent_3m": "{:,.0f}",
        "provisional_increase": "{:,.0f}",
        "provisional_new_limit": "{:,.0f}"
    })
)

Passed initial screen: 22
Candidates with a positive increase: 2
Total provisional additional limit: ₹450,000


,applicant_id,predicted_deterioration,current_credit_limit,average_balance_recent_3m,provisional_increase,provisional_new_limit
2460,SME005324,9.52%,"3,365,000","2,318,000","200,000","3,565,000"
2597,SME008138,8.76%,"1,635,000","1,228,433","250,000","1,885,000"


## 5. Retrieve Application PDs for the Pricing Scenario

We retrieve each candidate's original application PD from the saved
Milestone 2 predictions, matching by applicant ID.

Application PD estimates default risk. The early warning model estimates
deterioration risk; these probabilities are not interchangeable.

These application PDs use historical application information. They are
not refreshed default-risk estimates at month 23. We will use them only
as explicit proxies in an illustrative profitability scenario.

The early warning test split is separate from the application scorecard
split. Therefore, this is not an independent test of the entire lending
and limit-increase process.

In [5]:
scorecard_reports = root / "reports" / "sme_scorecard_v1"

pd_files = [
    scorecard_reports / "development_scored.csv",
    scorecard_reports / "validation_scored.csv"
]

pd_frames = []

for path in pd_files:
    if not path.is_file():
        raise FileNotFoundError(f"Missing saved predictions: {path}")

    frame = pd.read_csv(
        path,
        usecols=["applicant_id", "predicted_pd"],
        dtype={"applicant_id": "string"}
    )
    pd_frames.append(frame)

application_pds = (
    pd.concat(pd_frames, ignore_index=True)
    .rename(columns={"predicted_pd": "application_pd"})
)

assert application_pds["applicant_id"].notna().all()
assert application_pds["applicant_id"].is_unique
assert application_pds["application_pd"].between(0, 1).all()

# Safe to rerun: replace any previously joined application PD.
increase_candidates = (
    increase_candidates.drop(columns="application_pd", errors="ignore")
    .merge(
        application_pds,
        on="applicant_id",
        how="left",
        validate="one_to_one"
    )
)

assert increase_candidates["application_pd"].notna().all()

print(f"Candidate application PDs matched: {len(increase_candidates):,}")

display(
    increase_candidates[[
        "applicant_id",
        "application_pd",
        "predicted_deterioration",
        "provisional_increase"
    ]].style.format({
        "application_pd": "{:.2%}",
        "predicted_deterioration": "{:.2%}",
        "provisional_increase": "₹{:,.0f}"
    }).hide(axis="index")
)

Candidate application PDs matched: 2


applicant_id,application_pd,predicted_deterioration,provisional_increase
SME005324,7.45%,9.52%,"₹200,000"
SME008138,1.86%,8.76%,"₹250,000"


## 6. Assess Incremental Profitability

We evaluate only the proposed additional credit, using these assumptions:

| Assumption | Value |
|---|---:|
| Annual interest rate on additional borrowing | 16% |
| Average drawdown of the additional limit | 50% |
| Annual funding cost | 8% of additional drawn balance |
| Annual operating cost | 2% of additional drawn balance |
| Loss given default | 45% |
| Allocated equity | 10% of additional drawn balance |
| Minimum annual pre-tax return on allocated equity | 15% |

Calculations:
- Additional drawn balance = provisional increase × 50%
- Expected loss = application PD × 45% × additional drawn balance
- Annual profit = interest income − funding cost − operating cost − expected loss
- Return on allocated equity = annual profit ÷ allocated equity

Application PD is a historical proxy, not a refreshed month-23 PD.
Average drawn balance also serves as a simplified exposure-at-default proxy.

This scenario excludes taxes, capital against undrawn commitments,
default-related interest shortfalls, and changes in borrower behaviour.
The equity allocation is an assumption, not a regulatory capital calculation.

Passing the hurdle produces a provisional simulation offer, subject to
updated credit assessment. It does not establish actual profitability.

In [6]:
# Fixed assumptions: do not adjust them to force candidates to pass.
ANNUAL_INTEREST_RATE = 0.16
INCREMENTAL_DRAWDOWN = 0.50
FUNDING_COST_RATE = 0.08
OPERATING_COST_RATE = 0.02
LGD = 0.45
EQUITY_SHARE = 0.10
MIN_PRETAX_ROE = 0.15

candidate_economics = increase_candidates.copy()

candidate_economics["incremental_drawn_balance"] = (
    candidate_economics["provisional_increase"] * INCREMENTAL_DRAWDOWN
)

exposure = candidate_economics["incremental_drawn_balance"]

candidate_economics["annual_interest_income"] = (
    exposure * ANNUAL_INTEREST_RATE
)
candidate_economics["annual_funding_cost"] = (
    exposure * FUNDING_COST_RATE
)
candidate_economics["annual_operating_cost"] = (
    exposure * OPERATING_COST_RATE
)
candidate_economics["annual_expected_loss"] = (
    exposure * candidate_economics["application_pd"] * LGD
)
candidate_economics["allocated_equity"] = exposure * EQUITY_SHARE

candidate_economics["annual_profit_before_tax"] = (
    candidate_economics["annual_interest_income"]
    - candidate_economics["annual_funding_cost"]
    - candidate_economics["annual_operating_cost"]
    - candidate_economics["annual_expected_loss"]
)

assert candidate_economics["allocated_equity"].gt(0).all()

candidate_economics["incremental_pretax_roe"] = (
    candidate_economics["annual_profit_before_tax"]
    / candidate_economics["allocated_equity"]
)

candidate_economics["passes_profitability"] = (
    candidate_economics["incremental_pretax_roe"].ge(MIN_PRETAX_ROE)
)

candidate_economics["simulation_decision"] = np.where(
    candidate_economics["passes_profitability"],
    "Provisional offer — updated assessment required",
    "No offer — below profitability hurdle"
)

simulation_offers = candidate_economics.loc[
    candidate_economics["passes_profitability"]
].copy()

display(
    candidate_economics[[
        "applicant_id",
        "application_pd",
        "provisional_increase",
        "incremental_drawn_balance",
        "annual_expected_loss",
        "annual_profit_before_tax",
        "incremental_pretax_roe",
        "simulation_decision"
    ]].style.format({
        "application_pd": "{:.2%}",
        "provisional_increase": "₹{:,.0f}",
        "incremental_drawn_balance": "₹{:,.0f}",
        "annual_expected_loss": "₹{:,.0f}",
        "annual_profit_before_tax": "₹{:,.0f}",
        "incremental_pretax_roe": "{:.2%}"
    }).hide(axis="index")
)

print(f"Candidates assessed: {len(candidate_economics):,}")
print(f"Provisional simulation offers: {len(simulation_offers):,}")
print(
    "Total proposed additional limit: "
    f"₹{simulation_offers['provisional_increase'].sum():,.0f}"
)

applicant_id,application_pd,provisional_increase,incremental_drawn_balance,annual_expected_loss,annual_profit_before_tax,incremental_pretax_roe,simulation_decision
SME005324,7.45%,"₹200,000","₹100,000","₹3,355","₹2,645",26.45%,Provisional offer — updated assessment required
SME008138,1.86%,"₹250,000","₹125,000","₹1,047","₹6,453",51.63%,Provisional offer — updated assessment required


Candidates assessed: 2
Provisional simulation offers: 2
Total proposed additional limit: ₹450,000


## 7. Check Profitability Under Less Favourable Assumptions

Assess the same provisional increases under four illustrative scenarios:

| Scenario | Change from base |
|---|---|
| Base | Original assumptions |
| Higher default risk | Application PD multiplied by 1.5 |
| Higher funding cost | Funding cost rises from 8% to 10% |
| Combined downside | Both changes occur together |

The interest rate, increase amounts, drawdown, LGD, and equity allocation
remain unchanged.

These are sensitivity scenarios, not forecasts or a calibrated regulatory
stress test. We compare each scenario's pre-tax return on allocated equity
with the same 15% hurdle.

The base decisions are retained separately so that the downside results
remain visible.

In [7]:
scenario_settings = [
    ("Base", 1.0, FUNDING_COST_RATE),
    ("Higher default risk", 1.5, FUNDING_COST_RATE),
    ("Higher funding cost", 1.0, FUNDING_COST_RATE + 0.02),
    ("Combined downside", 1.5, FUNDING_COST_RATE + 0.02)
]

scenario_results = []

for scenario_name, pd_multiplier, funding_rate in scenario_settings:
    result = candidate_economics[[
        "applicant_id",
        "application_pd",
        "provisional_increase",
        "incremental_drawn_balance",
        "allocated_equity"
    ]].copy()

    result["scenario"] = scenario_name
    result["scenario_pd"] = (
        result["application_pd"] * pd_multiplier
    ).clip(upper=1)

    result["scenario_funding_rate"] = funding_rate

    result["scenario_expected_loss"] = (
        result["incremental_drawn_balance"]
        * result["scenario_pd"]
        * LGD
    )

    result["scenario_profit_before_tax"] = (
        result["incremental_drawn_balance"]
        * (
            ANNUAL_INTEREST_RATE
            - funding_rate
            - OPERATING_COST_RATE
        )
        - result["scenario_expected_loss"]
    )

    result["scenario_pretax_roe"] = (
        result["scenario_profit_before_tax"]
        / result["allocated_equity"]
    )

    result["clears_hurdle"] = (
        result["scenario_pretax_roe"].ge(MIN_PRETAX_ROE)
    )

    scenario_results.append(result)

sensitivity_results = pd.concat(
    scenario_results,
    ignore_index=True
)

# Confirm the base scenario reproduces the previous calculation.
base_check = (
    sensitivity_results.loc[
        sensitivity_results["scenario"].eq("Base"),
        ["applicant_id", "scenario_pretax_roe"]
    ]
    .merge(
        candidate_economics[[
            "applicant_id", "incremental_pretax_roe"
        ]],
        on="applicant_id",
        validate="one_to_one"
    )
)

assert np.allclose(
    base_check["scenario_pretax_roe"],
    base_check["incremental_pretax_roe"]
)

display(
    sensitivity_results[[
        "applicant_id",
        "scenario",
        "scenario_pd",
        "scenario_profit_before_tax",
        "scenario_pretax_roe",
        "clears_hurdle"
    ]].style.format({
        "scenario_pd": "{:.2%}",
        "scenario_profit_before_tax": "₹{:,.0f}",
        "scenario_pretax_roe": "{:.2%}"
    }).hide(axis="index")
)

print("Sensitivity check completed. Base decisions remain unchanged.")

applicant_id,scenario,scenario_pd,scenario_profit_before_tax,scenario_pretax_roe,clears_hurdle
SME005324,Base,7.45%,"₹2,645",26.45%,True
SME008138,Base,1.86%,"₹6,453",51.63%,True
SME005324,Higher default risk,11.18%,₹968,9.68%,False
SME008138,Higher default risk,2.79%,"₹5,930",47.44%,True
SME005324,Higher funding cost,7.45%,₹645,6.45%,False
SME008138,Higher funding cost,1.86%,"₹3,953",31.63%,True
SME005324,Combined downside,11.18%,"₹-1,032",-10.32%,False
SME008138,Combined downside,2.79%,"₹3,430",27.44%,True


Sensitivity check completed. Base decisions remain unchanged.


## 8. Export the Simulation Results and Assumptions

Save:
- Screening decisions for every assessed account.
- Candidate profitability calculations.
- Base-case provisional offers.
- Sensitivity results.
- Screening counts and policy assumptions.

A separate flag identifies candidates meeting the profitability hurdle
in every tested scenario. This flag does not change the base-case policy.

All proposals are synthetic simulations requiring updated credit assessment.
There are no observed post-increase outcomes to validate their success.

In [8]:
import json

export_directory = root / "reports" / "sme_limit_increase_v1"
export_directory.mkdir(parents=True, exist_ok=True)

# Record downside sensitivity without changing the base decisions.
scenario_resilience = (
    sensitivity_results.groupby("applicant_id")["clears_hurdle"]
    .all()
)

candidate_economics["passes_all_tested_scenarios"] = (
    candidate_economics["applicant_id"].map(scenario_resilience)
)

simulation_offers = candidate_economics.loc[
    candidate_economics["passes_profitability"]
].copy()

# Assign a decision to every assessed account.
decision_export = limit_review.copy()

decision_export["simulation_decision"] = np.where(
    ~decision_export["passes_initial_screen"],
    "No proposal — initial screen failed",
    "No proposal — insufficient headroom"
)

decision_map = candidate_economics.set_index(
    "applicant_id"
)["simulation_decision"]

candidate_mask = decision_export["applicant_id"].isin(decision_map.index)

decision_export.loc[candidate_mask, "simulation_decision"] = (
    decision_export.loc[candidate_mask, "applicant_id"].map(decision_map)
)

policy_assumptions = {
    "policy_version": "sme_limit_increase_v1",
    "method": "Rule-based synthetic simulation",
    "review_month_index": int(review_month),
    "screening": {
        "maximum_deterioration_probability": 0.10,
        "recent_utilization_minimum": 0.50,
        "recent_utilization_maximum": 0.85,
        "maximum_latest_utilization": 0.90,
        "maximum_recent_dpd": 0,
        "maximum_deposit_decline": 0.10,
        "maximum_recent_overdrafts": 2,
        "maximum_utilization_increase": 0.15
    },
    "sizing": {
        "target_utilization": TARGET_UTILIZATION,
        "maximum_increase_share": MAX_INCREASE_SHARE,
        "revenue_limit_share": REVENUE_LIMIT_SHARE,
        "round_down_to_rupees": ROUNDING_UNIT
    },
    "economics": {
        "annual_interest_rate": ANNUAL_INTEREST_RATE,
        "incremental_drawdown": INCREMENTAL_DRAWDOWN,
        "funding_cost_rate": FUNDING_COST_RATE,
        "operating_cost_rate": OPERATING_COST_RATE,
        "lgd": LGD,
        "allocated_equity_share": EQUITY_SHARE,
        "minimum_pretax_roe": MIN_PRETAX_ROE
    },
    "sensitivity_scenarios": [
        {
            "name": name,
            "pd_multiplier": multiplier,
            "funding_cost_rate": funding_rate
        }
        for name, multiplier, funding_rate in scenario_settings
    ],
    "limitations": [
        "Synthetic data; no observed limit-increase outcomes.",
        "Application PD and revenue are historical proxies.",
        "Early warning probability is not a default PD.",
        "Drawn balance is a simplified EAD proxy.",
        "Pre-tax economics exclude undrawn capital and interest shortfalls.",
        "The full decision pipeline has not been independently validated.",
        "Provisional offers require updated credit assessment."
    ]
}

tables_to_save = {
    "account_decisions": decision_export,
    "candidate_economics": candidate_economics,
    "provisional_offers": simulation_offers,
    "sensitivity_results": sensitivity_results,
    "screening_summary": screening_summary
}

for name, table in tables_to_save.items():
    table.to_csv(export_directory / f"{name}.csv", index=False)

assumptions_path = export_directory / "policy_assumptions.json"
assumptions_path.write_text(
    json.dumps(policy_assumptions, indent=2),
    encoding="utf-8"
)

# Verify the saved offers retain their IDs and amounts.
reloaded_offers = pd.read_csv(
    export_directory / "provisional_offers.csv",
    dtype={"applicant_id": "string"}
)

assert reloaded_offers["applicant_id"].tolist() == (
    simulation_offers["applicant_id"].tolist()
)

assert np.allclose(
    reloaded_offers["provisional_increase"],
    simulation_offers["provisional_increase"]
)

print("Export and reload checks passed.")
print(f"Accounts assessed: {len(decision_export):,}")
print(f"Base-case provisional offers: {len(simulation_offers):,}")
print(
    "Offers passing every tested scenario: "
    f"{simulation_offers['passes_all_tested_scenarios'].sum():,}"
)
print(f"Reports saved: {export_directory.relative_to(root)}")

Export and reload checks passed.
Accounts assessed: 2,967
Base-case provisional offers: 2
Offers passing every tested scenario: 1
Reports saved: reports/sme_limit_increase_v1
